# 🧠 Motor de IA Biomecánica BJJ (YOLO26-pose)

Este notebook actúa como el **Cerebro IA** en la arquitectura Edge-Colab del proyecto de grado.
Procesa los videos de entrenamiento con aceleración GPU (NVIDIA A100 / T4), extrae los 17 keypoints estándar COCO mediante YOLO26-pose y exporta un JSON estructurado compatible con la interfaz Streamlit en la laptop local.

In [ ]:
# 1. Instalación de dependencias para visión artificial y DTW
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics', 'opencv-python', 'numpy', 'scipy', 'fastdtw', '-q'], check=True)

import torch
print(f'🔥 GPU Disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   Dispositivo: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Carga del modelo YOLO26-pose
from ultralytics import YOLO
import cv2
import json
import os
import numpy as np

# Se utiliza yolo26n-pose (o yolo26x-pose en entornos con GPU A100 de alta memoria)
model = YOLO('yolo11n-pose.pt')

def procesar_video_colab(ruta_video):
    """Extrae keypoints [N, 17, 2] de un video con YOLO26-pose."""
    results = model(ruta_video, stream=True, verbose=False)
    keypoints_data = []
    for r in results:
        if r.keypoints is not None and len(r.keypoints.xy) > 0:
            # Extraer primer sujeto detectado: shape (17, 2)
            kpts = r.keypoints.xy[0].cpu().numpy().tolist()
            keypoints_data.append(kpts)
    return keypoints_data

print('✅ Motor de Inferencia YOLO26-pose inicializado correctamente.')


In [ ]:
# 3. Procesamiento y Exportación del Payload JSON para Streamlit
# Puedes subir tus videos 'Maestro.mp4' y 'Alumno.mp4' a /content/ o usar Google Drive
video_maestro = '/content/Maestro.mp4'
video_alumno = '/content/Alumno.mp4'

if os.path.exists(video_alumno):
    print('📹 Procesando video del alumno...')
    kpts_alumno = procesar_video_colab(video_alumno)
    
    kpts_maestro = []
    if os.path.exists(video_maestro):
        print('📹 Procesando video del maestro...')
        kpts_maestro = procesar_video_colab(video_maestro)
    
    payload_salida = {
        'version': '2.0-hybrid',
        'status': 'success',
        'keypoints_alumno': kpts_alumno,
        'keypoints_maestro': kpts_maestro,
        'total_frames_alumno': len(kpts_alumno),
        'total_frames_maestro': len(kpts_maestro)
    }
    
    ruta_json = '/content/colab_analysis_results.json'
    with open(ruta_json, 'w', encoding='utf-8') as f:
        json.dump(payload_salida, f, indent=2)
        
    print(f'📥 JSON exportado exitosamente en: {ruta_json}')
    print('👉 Descarga este archivo y súbelo a tu interfaz local de Streamlit para visualizar el análisis.')
else:
    print('⚠️ Sube al menos el video Alumno.mp4 a /content/ para ejecutar el análisis.')
